# Lab 1 Part 1 — The Modern GenAI Stack
**Day 1 Morning | ~45 minutes | Colab CPU**

---

## What You Will Build
By the end of Part 1 you will have:
1. Unwrapped a HuggingFace `pipeline` to see tokens → logits → probabilities
2. Called a hosted LLM (GPT-4o-mini) with the OpenAI client
3. Streamed tokens live to your terminal
4. Swapped the **same client code** to a different provider by changing one line
5. Chained a prompt template with LangChain

> **The key idea:** The OpenAI Chat Completions format became the de-facto wire protocol.
> Change `base_url` — keep everything else. This composes the whole course.

In [ ]:
!uv pip install -q "transformers>=5" torch sentence-transformers openai langchain langchain-openai httpx
print('Install complete')


## 🔑 Configuration
Add the instructor-provided OpenAI API key to Colab Secrets as `OPENAI_API_KEY`. Everything in this lab flows from this configuration cell.

In [ ]:
# Configuration — works in Colab Secrets or local environment variables
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, open the key icon in the left sidebar, "
            f"add a secret named {name}, paste the value, and enable Notebook access."
        )
    return value

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
OPENAI_BASE_URL = 'https://api.openai.com/v1'
DEFAULT_MODEL   = 'gpt-4o-mini'
QUALITY_MODEL   = 'gpt-4o'

print(f'Config loaded — OPENAI_API_KEY starts with {OPENAI_API_KEY[:8]}...')


---

## HuggingFace API Tiers

HuggingFace gives you several levels of control. Part A walks down the stack so you can choose the right API for the job.

| Level | API | Best for | What you trade away |
| --- | --- | --- | --- |
| High-level | `pipeline()` | Fast prototypes and task demos | Less control over tokens, batching, and internals |
| Mid-level | `AutoTokenizer` + `AutoModelForCausalLM` | Custom prompts, generation settings, batching, inspection | More boilerplate |
| Low-level | Direct model class, custom `generate()`, custom kernels | Research, architecture work, advanced deployment | You own more details |
| Serving | `transformers serve` | Local/self-hosted OpenAI-compatible endpoint | Experimental relative to vLLM/TGI/SGLang at scale |

In Transformers v5, this map matters even more: HuggingFace sharpened the serving story with `transformers serve`, and quantized loading is configured through `quantization_config` rather than older shortcut flags.


---

## Part A — HuggingFace From High-Level to Low-Level (15 min)

We start local and small: a 124 M-parameter GPT-2 model that runs on Colab CPU. The point is not the quality — the point is to **open the black box**.

You will see three views of the same model:

1. `pipeline()` for task-oriented inference.
2. `AutoTokenizer` + `AutoModelForCausalLM.generate()` for controlled generation.
3. A direct forward pass that exposes logits and next-token probabilities.


* ⚠️ STUDENT NOTE: You may see a warning about `h.{0...11}.attn.bias` marked as UNEXPECTED.
This is safe to ignore. These are causal mask buffers (not learnable weights) from the original OpenAI GPT-2 checkpoint that don't map exactly to the HuggingFace model class. The model loads correctly — if you got coherent text output, everything worked.


In [ ]:
# Cell A0 — Check the Transformers version
import transformers
print(f'Transformers version: {transformers.__version__}')
print('v5 note: AutoTokenizer chooses the best tokenizer backend automatically; the old fast/slow distinction is no longer something students usually need to manage.')


In [ ]:
# Cell A1 — The simplest possible inference: 3 lines
# INSTRUCTOR NOTE: pause here. 'This is the entire AI industry abstracted into 3 lines.'
from transformers import pipeline, GenerationConfig

generator = pipeline('text-generation', model='gpt2')
gen_config = GenerationConfig(max_new_tokens=40)

result = generator('The best way to deploy a language model is', generation_config=gen_config)
print(result[0]['generated_text'])

### Mid-Level: `AutoTokenizer` + `AutoModelForCausalLM`

`pipeline()` is convenient, but production code often needs more control: exact tokenization, prompt formatting, generation settings, batching, and access to outputs. The `Auto*` classes are the normal middle layer. They load the correct tokenizer/model class from the checkpoint metadata without you importing an architecture-specific class.


In [ ]:
# Cell A1b — Mid-level generation with AutoTokenizer + AutoModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

inputs = tokenizer('The best way to deploy', return_tensors='pt')
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


In [ ]:
# Cell A2 — Unwrap the mid-level call: text → token IDs
# INSTRUCTOR NOTE: 'Let us open the black box. Three steps happen every time.'
# Reuse model_id, tokenizer, and model from Cell A1b. If you skipped it, uncomment the two lines below.
# tokenizer = AutoTokenizer.from_pretrained('gpt2')
# model = AutoModelForCausalLM.from_pretrained('gpt2')

text   = 'The best way to deploy a language model is'
inputs = tokenizer(text, return_tensors='pt')

print('STEP 1 — TEXT → TOKENS')
print(f'  Input text : {text!r}')
print(f'  Token IDs  : {inputs["input_ids"].tolist()[0]}')
print(f'  Decoded    : {[tokenizer.decode([t]) for t in inputs["input_ids"][0]]}')
print(f'  Count      : {inputs["input_ids"].shape[1]} tokens')


In [ ]:
# Cell A3 — token IDs → logits → next-token probabilities
with torch.no_grad():
    outputs = model(**inputs)

logits          = outputs.logits
next_token_logits = logits[0, -1, :]          # predictions for the *next* token
probs           = torch.softmax(next_token_logits, dim=-1)
top5            = torch.topk(probs, 5)

print('STEP 2 — MODEL PREDICTS NEXT TOKEN')
print(f'  Vocab size : {logits.shape[-1]:,}')
print()
print('  Top 5 candidates:')
for score, idx in zip(top5.values, top5.indices):
    print(f'    {tokenizer.decode([idx.item()])!r:<20} {score.item()*100:.1f}%')

In [ ]:
# Cell A4 — Memory math: why model size matters for deployment
# INSTRUCTOR NOTE: connect this to the Memory slide (S053).

def memory_table(model, label):
    params  = sum(p.numel() for p in model.parameters())
    fp32_mb = params * 4 / 1e6
    fp16_mb = params * 2 / 1e6
    int4_mb = params * 0.5 / 1e6
    print(f'\n{label}')
    print(f'  Parameters : {params:,}  ({params/1e6:.0f}M)')
    print(f'  FP32       : {fp32_mb:.0f} MB')
    print(f'  FP16       : {fp16_mb:.0f} MB')
    print(f'  INT4       : {int4_mb:.0f} MB')
    print(f'  Scale-up   → a 7B model is FP16 ~14 GB, INT4 ~3.5 GB')

memory_table(model, 'GPT-2 (124M params)')

### Transformers v5 Quantization Note

The memory table is arithmetic. Actually loading quantized models uses a configuration object in modern Transformers:

```python
from transformers import BitsAndBytesConfig

qconfig = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=qconfig,
    device_map='auto',
)
```

Older examples on the internet may show `load_in_4bit=True` directly inside `from_pretrained()`. Treat those as legacy snippets. Lab 4 uses the modern `quantization_config` pattern.


### Serving Bridge: `transformers serve`

Transformers v5 includes a lightweight local serving CLI. It starts an OpenAI-compatible server for HuggingFace models, which connects Part A directly to Part B's `base_url` idea.

Example terminal commands outside this notebook:

```bash
pip install "transformers[serving]>=5"
transformers serve --force-model Qwen/Qwen2.5-0.5B-Instruct --port 8000
```

Then the same OpenAI client pattern can point at the local server:

```python
client = OpenAI(api_key='not-needed', base_url='http://localhost:8000/v1')
```

For high-scale production, HuggingFace still points learners toward engines such as vLLM, SGLang, or TGI. For classroom experiments and local evaluation, `transformers serve` is a very useful bridge.


---

## Part B — Cloud LLM via the OpenAI Client (15 min)

> **The key pattern** — learn this once and it works with OpenAI, Groq, vLLM, 
> HuggingFace Inference Providers, LiteLLM, and the custom server you will build in Lab 4.
>
> ```python
> client = OpenAI(api_key=..., base_url=...)  # ← the only thing that changes
> ```

### The Chat Completions Wire Format

The OpenAI client is a Python wrapper around an HTTP API. The important request body is a list of `messages`:

- `system`: high-priority behavior instructions.
- `user`: the user's request.
- `assistant`: previous model responses when you continue a conversation.
- `tool`: tool results, which you will use in Part 2.

Order matters. A chat request is not just one prompt string; it is an ordered conversation transcript. The server URL (`base_url`) points to an API that understands this schema.

In the response, watch three fields in particular:

- `choices[0].message.content`: the text answer.
- `choices[0].finish_reason`: why generation stopped (`stop`, `length`, `tool_calls`, etc.).
- `usage`: token accounting for cost and latency analysis.


In [ ]:
# Cell B1 — The canonical OpenAI client call
# INSTRUCTOR NOTE: 'One pattern. Infinite backends. This line unlocks the whole course.'
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

messages = [
    {'role': 'system', 'content': 'You are an LLM deployment expert. Be concise.'},
    {'role': 'user',   'content': 'What are the top 3 reasons LLM demos fail in production?'}
]

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages
)
print(response.choices[0].message.content)
print(f'\nfinish_reason: {response.choices[0].finish_reason}')
print(f'Tokens used  : {response.usage.total_tokens}')


### Inspect the Raw Response Object

Before we build more abstractions, inspect the response object. This is the protocol shape that OpenAI-compatible servers return. Later, your FastAPI server and tool-calling loop will need to produce or consume the same kinds of fields.


In [ ]:
# Cell B1b — Inspect the response object as a plain dictionary
raw = response.model_dump()

print('Top-level fields:', list(raw.keys()))
print('Response id      :', raw['id'])
print('Model            :', raw['model'])
print('Finish reason    :', raw['choices'][0]['finish_reason'])
print('Message role     :', raw['choices'][0]['message']['role'])
print('Usage            :', raw['usage'])
print()
print('Full first choice:')
raw['choices'][0]


In [ ]:
# Improve output formatting
from IPython.display import display, Markdown
display(Markdown(response.choices[0].message.content))

In [ ]:
# Cell B2 — Streaming: watch tokens arrive live
# INSTRUCTOR NOTE: slow down here. Let everyone watch the tokens appear one by one.
print('Streaming response (tokens appear as generated):')
print('─' * 60)

stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'Explain the difference between FP16 and INT4 in 4 bullet points.'}],
    stream=True
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print('\n' + '─' * 60)

In [ ]:
# same but with improved formatting
from IPython.display import display, Markdown, update_display

print('Streaming response (tokens appear as generated):')
print('─' * 60)

stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'Explain the difference between FP16 and INT4 in 4 bullet points.'}],
    stream=True
)

full_response = ''
display_handle = display(Markdown(''), display_id=True)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        full_response += delta
        update_display(Markdown(full_response), display_id=display_handle.display_id)

print('─' * 60)

In [ ]:
# Cell B3 — Quality comparison: same question, two models
# INSTRUCTOR NOTE: 'Change only the model name. The client does not care.'
question = 'In one sentence: what is the most important thing to know about LLM serving?'

r_mini = client.chat.completions.create(
    model=DEFAULT_MODEL, messages=[{'role': 'user', 'content': question}]
)
r_full = client.chat.completions.create(
    model=QUALITY_MODEL,  messages=[{'role': 'user', 'content': question}]
)

print(f'gpt-4o-mini : {r_mini.choices[0].message.content}')
print(f'gpt-4o      : {r_full.choices[0].message.content}')

In [ ]:
# Cell B4 — *** THE KEY MOMENT: swap base_url, keep everything else ***
# INSTRUCTOR NOTE: pause. 'Notice what changed: one string. Nothing else.'
# This also works with:
#   Groq:        base_url='https://api.groq.com/openai/v1', api_key='gsk_...'
#   vLLM:        base_url='http://your-gpu-server:8000/v1',  api_key='not-needed'
#   Lab 5 server:base_url='https://your-ngrok-url/v1',       api_key='not-needed'
#   LiteLLM GW:  base_url='http://gateway:4000/v1',          api_key='not-needed'

GROQ_BASE_URL   = 'https://api.groq.com/openai/v1'
GROQ_API_KEY    = 'gsk_PASTE_GROQ_KEY_IF_AVAILABLE'  # optional — demo only
GROQ_MODEL      = 'llama-3.1-8b-instant'

providers = {
    'OpenAI gpt-4o-mini': {
        'base_url': OPENAI_BASE_URL, 'api_key': OPENAI_API_KEY, 'model': DEFAULT_MODEL
    },
    # Uncomment below if instructor provides a Groq key:
    # 'Groq llama-3.1-8b': {
    #     'base_url': GROQ_BASE_URL, 'api_key': GROQ_API_KEY, 'model': GROQ_MODEL
    # },
}

q = 'In one sentence: what is PagedAttention?'
for name, cfg in providers.items():
    c = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])
    r = c.chat.completions.create(model=cfg['model'], messages=[{'role': 'user', 'content': q}])
    print(f'[{name}]\n  {r.choices[0].message.content}\n')

---

## Part C — LangChain (15 min)

So far, you have called the API directly. That is the right baseline. Frameworks like LangChain become useful when your app grows beyond one-off calls. They add prompt templates, output parsers, composition, retries, tracing hooks, and integrations — at the cost of another abstraction layer.

The goal here is not to memorize LangChain. The goal is to see how the same OpenAI-compatible backend can sit underneath a framework. Same `base_url` pattern, higher-level app structure.


In [ ]:
# Cell C1 — ChatOpenAI with our Groq/OpenAI backend
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=DEFAULT_MODEL
)

messages = [
    SystemMessage(content='You are an LLM deployment expert. Be concise.'),
    HumanMessage(content='What is the difference between vLLM and a simple FastAPI server for LLM inference?')
]
print(llm.invoke(messages).content)

In [ ]:
# Cell C2 — Prompt templates + chaining: the LCEL pipe operator
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_messages([
    ('system', 'You are a technical instructor. Explain concepts clearly with one analogy.'),
    ('user',   'Explain {concept} to someone who {background}.')
])

chain = template | llm | StrOutputParser()

print(chain.invoke({
    'concept':    'quantization',
    'background': 'has never worked with neural networks before'
}))

### Conceptual Checkpoint

You have now seen the same deployment story at three layers:

1. **Local model layer:** HuggingFace `pipeline()` hides tokenization and generation, then you opened it up.
2. **Hosted API layer:** Chat Completions sends ordered `messages` to an OpenAI-compatible endpoint and returns `choices`, `finish_reason`, and `usage`.
3. **Framework layer:** LangChain wraps the same backend with templates and parsers so larger apps are easier to compose.

Part 2 adds one more capability: instead of only returning text, the model can request that your code run a tool.


---

## ✅ Lab 1 Part 1 Complete

You should now have:
- [ ] GPT-2 token IDs and top-5 next-token probabilities printed
- [ ] Memory table for GPT-2 (FP32 / FP16 / INT4)
- [ ] A streaming response from GPT-4o-mini
- [ ] Two-model quality comparison
- [ ] LangChain chain producing an analogy

## Stretch Goals

1. **Token counting:** Call the same prompt at three temperatures (0, 0.5, 1.0). 
   What changes? What stays the same?
2. **Browse the Hub:** Go to `huggingface.co/models?pipeline_tag=text-generation`. 
   Find a model fine-tuned for SQL generation. What is its parameter count?
3. **Logprobs:** Add `logprobs=True` to a non-streaming OpenAI call. 
   Print the top-5 token probabilities for the first word of the response.
4. **Add Groq:** If the instructor provides a Groq key, uncomment the Groq provider 
   in Cell B4. Compare speed vs GPT-4o-mini on the same prompt.
5. **Memory math:** Compute memory estimates for a 7B, 13B, and 70B model at FP16 and INT4. 
   Which fits in a T4 GPU (16 GB)?
6. **Transformers serve:** After Part B, run `transformers serve --force-model Qwen/Qwen2.5-0.5B-Instruct --port 8000` in a terminal. Replace `OPENAI_BASE_URL` with `http://localhost:8000/v1`. Does the same client code work?

## Next: Lab 1 Part 2

Part 1 treated the LLM as an inference engine: prompt in, text out. Part 2 changes the mental model. The LLM can request actions through tools/functions, observe the result, and then answer with grounded information.
